In [ ]:
## Streamlined methods to prepare weight categories for a HAF weighted analysis, 
## meant to be used after running the sdd_clip function in R

#Author: Kaitlin Lubetkin
#Created: 9/24/21
#Last Edited: 4/21/23


In [ ]:
## Before running, make sure your Arc Pro map has the following:
# A layer group called "GRSG_boundaries" that has
#     the fine-scale boundary (labeled "[FineScale]_FineScale_Boundary")
#     S-3/spring Seasonal Use Area polygon (labeled "[FineScale]_Spring_SUA")
#     S-4/summer Seasonal Use Area polygon (labeled "[FineScale]_Summer_SUA")
#     S-6/spring Seasonal Use Area polygon (labeled "[FineScale]_Winter_SUA")
# A layer group called "[FineScale]_SDD" that has the TerrADat Sample Design Database featureclasses
#     TerrestrialSamplePoints
#     TerrestrialPointEvaluation
#     TerrestrialSampleFrame
#     TerrestrialStrata
# A layer group called "RATINGS_gdb" that has the point featureclasses from the tool 1 output
#     [FineScale]_HAF
#     [FineScale]_S3_Spring
#     [FineScale]_S4_Summer
#     [FineScale]_S6_Winter
# A layer group called "[FineScale]_LMF_strata_segments" that has
#     strata_[FineScale]
#     segment_[FineScale]
#     segment_[FineScale]_spring
#     segment_[FineScale]_summer
#     segment_[FineScale]_winter

In [1]:
## Prep workspace and define functions
import os
import arcpy

map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]

# Union all strata shapefiles created using the sdd_clip function in R
def union_strata(season):
    frame_list = []
    frame_path = frame_path_base + season[0].upper() + season[1:] + "_SUA"
    
    # List sample frame names - choose from below and ajust as needed depending on how many lmf strata are included, 
    # and how many sample frames (if any) are included 
    #name_list = ["lmf1", "lmf2", "lmf3", "lmf4"] + ["s" + str(i) for i in range(1, maxS[season] + 1)]
    #name_list = ["lmf1", "lmf2"] + ["s" + str(i) for i in range(1, maxS[season] + 1)]
    name_list = ["lmf1"] + ["s" + str(i) for i in range(1, maxS[season] + 1)] 
    
    # can now just run straight through the rest
    for file in os.listdir(frame_path):
        if file.endswith(".shp"):
            f = frame_path + "\\" + file
            frame_list.append(f)
    
    arcpy.analysis.Union(frame_list, analysis_gdb + "\\tempUNION", "NO_FID")
    arcpy.management.RepairGeometry("tempUNION", "DELETE_NULL")
    #print("geometry repaired")
    
    arcpy.management.AddField("tempUNION", "area_hectares", "DOUBLE")
    arcpy.management.CalculateGeometryAttributes("tempUNION", 
                                                 "area_hectares AREA", 
                                                 area_unit = "HECTARES")
    #print("area calculated")
    arcpy.management.AddField("tempUNION", "strata_combo", "TEXT", field_length = 80)
    arcpy.management.AddField("tempUNION", "Design_Stratum", "TEXT", field_length = 255)
    addr = ["!id_" + n + "!" for n in name_list]
    arcpy.management.CalculateField("tempUNION", "strata_combo", 
                                    "ConcatAddr(" + ','.join([a for a in addr]) + ")", 
                                    "PYTHON3", 
                                    "def ConcatAddr(*args): return ''.join([str(i) for i in args if i not in(None,' ')]) ", 
                                    "TEXT", 
                                    "NO_ENFORCE_DOMAINS")
    #print("strata combo populated")
    arcpy.management.SelectLayerByAttribute("tempUNION", "NEW_SELECTION", "area_hectares >= 1.5", None)
    addr = ["!name_" + n + "!" for n in name_list]
    arcpy.management.CalculateField("tempUNION", "Design_Stratum", 
                                    "ConcatAddr(" + ','.join([a for a in addr]) + ")", 
                                    "PYTHON3", 
                                    "def ConcatAddr(*args): return ''.join([str(i) for i in args if i not in(None,' ')]) ", 
                                    "TEXT", 
                                    "NO_ENFORCE_DOMAINS")
    arcpy.management.SelectLayerByAttribute("tempUNION", "CLEAR_SELECTION")
    fm =('Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,tempUNION,Shape_Length,-1,-1;' + 
         'Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,tempUNION,Shape_Area,-1,-1;' + 
         'area_hectares "area_hectares" true true false 8 Double 0 0,First,#,tempUNION,area_hectares,-1,-1;' + 
         'strata_combo "strata_combo" true true false 80 Text 0 0,First,#,tempUNION,strata_combo,0,80;' + 
         'Design_Stratum "Design_Stratum" true true false 255 Text 0 0,First,#,tempUNION,Design_Stratum,0,255')
    arcpy.conversion.FeatureClassToFeatureClass("tempUNION", analysis_gdb, "strata_" + season + "_UNION", 
                                                field_mapping=fm)
    arcpy.management.Delete(analysis_gdb + "\\tempUNION")
        
# Prepare dissolved version and calculate weight categories
def dissolve_strata(season):
    #Dissolve based on the "strata_combo" & "Design_Stratum"
    arcpy.management.Dissolve("strata_" + season + "_UNION", 
                              analysis_gdb + "\\strata_" + season + "_UNION_Dissolve", 
                              "strata_combo;Design_Stratum")
    #Calculate hectares
    arcpy.management.CalculateGeometryAttributes("strata_" + season + "_UNION_Dissolve", 
                                                 "area_hectares_" + season + " AREA", 
                                                 area_unit = "HECTARES")
    #Add weight cat field
    arcpy.management.CalculateField("strata_" + season + "_UNION_Dissolve", 
                                    "wgtcat_" + season, 
                                    "!OBJECTID!", "PYTHON3",
                                    field_type="SHORT")

In [2]:
# parent folder, which has subfolders "1_Analysis", "2_Methods", and "3_Results"
parent_folder = r'C:\Users\ebouchard\Documents\WamboltCreekLakeValley'

# base path for all of the strata frames
frame_path_base = parent_folder + '\\2_Methods\\WamboltCreek_LakeValley_'

# location of the analysis inputs geodatabase
analysis_gdb = parent_folder + "\\1_Analysis\\" + 'WamboltCreek_LakeValley_SiteScale_Analysis_Inputs_20230928.gdb'

# name of the fine scale (should match the name used in the layer groups, and in the file names)
fine_scale = "WamboltCreek_LakeValley"


# The maximum TerrADat stratum number (will be 2 less than the max sXX.shp if the 2 LMF strata are included)
# Will need to go through the frame shapefiles in 2_Methods\Season to see what the maximum "s#" is
#maxS = {'spring': 10, 'summer': 11, 'winter': 11}


# Union all strata for each season
#for thisSeason in ["winter", "summer", "spring"]:
   # union_strata(thisSeason)
#del thisSeason

In [ ]:
## Next, manually fix slivers (<1 ha) by giving them the strata_combo and Design_Stratum of the adjacent polygon

# also double check very small non-quite-slivers (1-1.5 ha) and give own name or split as 
# needed to merge with adjacent larger polygons

# Then proceed with next Jupyter cell

In [7]:
## Dissolve each season - can run straight through this, no edits needed
#dissolve_strata("winter")
#dissolve_strata("summer")
dissolve_strata("spring")

In [ ]:
## Double check small (1-2 ha) polygons
## If necessary, return to the UNION and reassign strata combo & sample design, then redo the Dissolve

In [8]:
## Add weight cats from UNION_Dissolve to the SDD points - can run straight through this, no edits needed
for thisSeason in ["_spring", "_summer", "_winter"]:
    arcpy.management.AddField(fine_scale + r"_SDD\TerrestrialPointEvaluation", "wgtcat" + thisSeason, "SHORT")
    arcpy.analysis.SpatialJoin(fine_scale + r"_SDD\TerrestrialPointEvaluation", 
                               "strata" + thisSeason + "_UNION_Dissolve", 
                               analysis_gdb + "\\TerrPt_spjoin" + thisSeason)
    arcpy.management.AddJoin(fine_scale + r"_SDD\TerrestrialPointEvaluation", "TerrestrialVisitID", 
                             "TerrPt_spjoin" + thisSeason, "TerrestrialVisitID", "KEEP_ALL")
    arcpy.management.CalculateField(fine_scale + r"_SDD\TerrestrialPointEvaluation", 
                                    "TerrestrialPointEvaluation.wgtcat" + thisSeason, 
                                    "!TerrPt_spjoin" + thisSeason + ".wgtcat" + thisSeason + "_1!", "PYTHON3")
    arcpy.management.RemoveJoin(fine_scale + r"_SDD\TerrestrialPointEvaluation", "TerrPt_spjoin" + thisSeason)
    arcpy.management.Delete(analysis_gdb + "\\TerrPt_spjoin" + thisSeason)
    
## Add blank LMF-related fields to the SDD points
arcpy.management.AddField(fine_scale + r"_SDD\TerrestrialPointEvaluation", "segcode", "TEXT", 20)
for thisSeason in ["_spring", "_summer", "_winter"]:
    arcpy.management.AddField(fine_scale + r"_SDD\TerrestrialPointEvaluation", "area_ratio" + thisSeason, "DOUBLE")     
del thisSeason
## If any SDD points fall in a LMF segment, will need to manually populate those fields
arcpy.management.SelectLayerByLocation(in_layer=fine_scale + r"_SDD\TerrestrialPointEvaluation", 
                                       overlap_type="INTERSECT", 
                                       select_features=fine_scale + r"_LMF_strata_segments\segment_" + fine_scale, 
                                       selection_type="NEW_SELECTION")
result = arcpy.management.GetCount(fine_scale + r"_SDD\TerrestrialPointEvaluation")
if int(result[0]) > 0:
    print("{} TerrestrialPointEvaluation plots fall in an LMF segment".format(result[0]))
    print("Should manually populate segment_id and area_ratio from the relevant segement_finescale_season layer(s).")

In [9]:
## Project points from RATINGS gdb into Albers Equal Area and save in the analysis inputs gdb
# project name from the tool 1 output gdb
base_name = "WamboltCreek_LakeValley"

# can just run straight through the rest - no edits needed
albers = 'PROJCS["NAD_1983_Albers",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0],UNIT["Meter",1.0]]'
arcpy.management.Project("RATINGS_gdb\\" + base_name + "_HAF", 
                         analysis_gdb + "\\" + fine_scale + "_HAF", 
                         albers)
# Run each season
for thisSeason in [["_S6_Winter", "_winter"], ["_S4_Summer", "_summer"], ["_S3_Spring", "_spring"]]:
#for thisSeason in [["_S6_Winter", "_winter"]]:
    arcpy.management.Project("RATINGS_gdb\\" + base_name + thisSeason[0], 
                             analysis_gdb + "\\" + fine_scale + thisSeason[0], 
                             albers)
    # Split out TerrADat points
    arcpy.management.SelectLayerByAttribute(fine_scale + thisSeason[0], "NEW_SELECTION", "ProjectName<>'LMF'", None)
    #arcpy.management.SelectLayerByAttribute(fine_scale + thisSeason[0], "NEW_SELECTION", "Source<>'LMF'", None)
    arcpy.management.CopyFeatures(fine_scale + thisSeason[0], analysis_gdb + "\\tdat_" + fine_scale + thisSeason[1])
    # Split out LMF points
    arcpy.management.SelectLayerByAttribute(fine_scale + thisSeason[0], "NEW_SELECTION", "ProjectName='LMF'", None)
    #arcpy.management.SelectLayerByAttribute(fine_scale + thisSeason[0], "NEW_SELECTION", "Source='LMF'", None)
    arcpy.management.CopyFeatures(fine_scale + thisSeason[0], analysis_gdb + "\\lmf_" + fine_scale + thisSeason[1])
    # Add fields for weight cat, segment, and area_ratio
    arcpy.management.AddField("tdat_" + fine_scale + thisSeason[1], "wgtcat" + thisSeason[1], "SHORT")
    arcpy.management.AddField("tdat_" + fine_scale + thisSeason[1], "segcode", "TEXT", 20)
    arcpy.management.AddField("tdat_" + fine_scale + thisSeason[1], "area_ratio" + thisSeason[1], "DOUBLE")
    arcpy.management.AddField("lmf_" + fine_scale + thisSeason[1], "wgtcat" + thisSeason[1], "SHORT")
    arcpy.management.AddField("lmf_" + fine_scale + thisSeason[1], "segcode", "TEXT", 20)
    arcpy.management.AddField("lmf_" + fine_scale + thisSeason[1], "area_ratio" + thisSeason[1], "DOUBLE")
    # Join TerrADat points to PointEvaluation fc and calculate additional fields
    arcpy.management.AddJoin("tdat_" + fine_scale + thisSeason[1], "PrimaryKey",
                             fine_scale + r"_SDD\TerrestrialPointEvaluation", "TerrADatPrimaryKey", 
                             "KEEP_ALL")
    arcpy.management.CalculateField("tdat_" + fine_scale + thisSeason[1], 
                                    "wgtcat" + thisSeason[1], 
                                    "!TerrestrialPointEvaluation.wgtcat" + thisSeason[1] + "!", "PYTHON3")
    arcpy.management.CalculateField("tdat_" + fine_scale + thisSeason[1], 
                                    "segcode", 
                                    "!TerrestrialPointEvaluation.segcode!", "PYTHON3")
    arcpy.management.CalculateField("tdat_" + fine_scale + thisSeason[1], 
                                    "area_ratio" + thisSeason[1], 
                                    "!TerrestrialPointEvaluation.area_ratio" + thisSeason[1] + "!", "PYTHON3")
    arcpy.management.RemoveJoin("tdat_" + fine_scale + thisSeason[1], "TerrestrialPointEvaluation")
    
    # Spatially join LMF points to dissolved seasonal strata and calculate weight cat
    arcpy.analysis.SpatialJoin("lmf_" + fine_scale + thisSeason[1], 
                               "strata" + thisSeason[1] + "_UNION_Dissolve", 
                               analysis_gdb + "\\LMFseg_spjoin" + thisSeason[1])
    arcpy.management.AddJoin("lmf_" + fine_scale + thisSeason[1], "PlotKey", 
                             "LMFseg_spjoin" + thisSeason[1], "PlotKey", "KEEP_ALL")
    arcpy.management.CalculateField("lmf_" + fine_scale + thisSeason[1], 
                                    "wgtcat" + thisSeason[1], 
                                    "!LMFseg_spjoin" + thisSeason[1] + ".wgtcat" + thisSeason[1] + "_1!", "PYTHON3")
    arcpy.management.RemoveJoin("lmf_" + fine_scale + thisSeason[1], "LMFseg_spjoin" + thisSeason[1])
    arcpy.management.Delete(analysis_gdb + "\\LMFseg_spjoin" + thisSeason[1])
    
    # Spatially join LMF points to seasonal segments and calculate segment_id & area_ratio
    arcpy.analysis.SpatialJoin("lmf_" + fine_scale + thisSeason[1], 
                               fine_scale + r"_LMF_strata_segments\segment_" + fine_scale + thisSeason[1], 
                               analysis_gdb + "\\LMFseg_spjoin" + thisSeason[1])
    arcpy.management.AddJoin("lmf_" + fine_scale + thisSeason[1], "PlotKey", 
                             "LMFseg_spjoin" + thisSeason[1], "PlotKey", "KEEP_ALL")
    arcpy.management.CalculateField("lmf_" + fine_scale + thisSeason[1], 
                                    "segcode", 
                                    "!LMFseg_spjoin" + thisSeason[1] + ".segcode_1!", "PYTHON3")
    arcpy.management.CalculateField("lmf_" + fine_scale + thisSeason[1], 
                                    "area_ratio" + thisSeason[1], 
                                    "!LMFseg_spjoin" + thisSeason[1] + ".area_ratio!", "PYTHON3")
    arcpy.management.RemoveJoin("lmf_" + fine_scale + thisSeason[1], "LMFseg_spjoin" + thisSeason[1])
    arcpy.management.Delete(analysis_gdb + "\\LMFseg_spjoin" + thisSeason[1])
    
print("Done!")

Done!


In [ ]:
## Double check that none of the seasonal tdat or lmf points has NULL for wgt cat
## Then, return to the seasonal rmd scripts